In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score

from sklearn.svm import SVC
from tqdm import tqdm  # Provides the progress of model running
import os

In [2]:
hr = pd.read_csv('D:/Machine_Learning/Cases/HR_comma_sep.csv')

X , y = hr.drop('left', axis=1), hr['left']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26, stratify= hr['left'])

In [3]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import  ColumnTransformer
from sklearn.compose import make_column_selector

ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(transformers=[("OHE", ohe, make_column_selector(dtype_include=object))], remainder="passthrough",
                            verbose_feature_names_out=False).set_output(transform="pandas")

scaler = StandardScaler()

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)
X_trn_scl = scaler.fit_transform(X_trn_ohe)
X_tst_scl = scaler.transform(X_tst_ohe)

#### Linear Kernel

In [4]:
Cs=np.linspace(0.001,5,20)
scores=[]
for c in tqdm(Cs):
    svm=SVC(kernel='linear', probability=True, random_state=26, C=c)
    svm.fit(X_trn_scl, y_train)
    
    y_pred_prob = svm.predict_proba(X_tst_scl)
    
    scores.append([c, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['C', 'log_loss' ])
df_scores.sort_values(['log_loss'], ascending=True).head()

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [09:42<00:00, 29.15s/it]


,C,log_loss
11,2.895158,0.437987
8,2.105842,0.437995
18,4.736895,0.437996
13,3.421368,0.438002
15,3.947579,0.438004


#### Polynomial Kernel - Takes too much time

In [5]:
# Cs=np.linspace(0.001,5,10)
# Ds=[2,3,4]
# scores=[]

# for c in tqdm(Cs):
#     for d in Ds:
#         svm=SVC(kernel='poly', probability=True, random_state=26, C=c, degree=d)
#         svm.fit(X_trn_scl, y_train)
        
#         y_pred_prob = svm.predict_proba(X_tst_scl)
        
#         scores.append([c, d, log_loss(y_test, y_pred_prob)])
    
# df_scores = pd.DataFrame(scores, columns=['C', 'Degree', 'Log_Loss' ])
# df_scores.sort_values(['Log_Loss'], ascending=True).head()

#### RBF(Radial) Kernel

In [ ]:
Cs=np.linspace(0.001,5,10)
Gs=np.linspace(0.001,5,10)
scores=[]

for c in tqdm(Cs):
    for g in Gs:
        svm=SVC(kernel='rbf', probability=True, random_state=26, C=c, gamma=g)
        svm.fit(X_trn_scl, y_train)
        
        y_pred_prob = svm.predict_proba(X_tst_scl)
        
        scores.append([c, g, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['C', 'Gamma', 'Log_Loss' ])
df_scores.sort_values(['Log_Loss'], ascending=True).head()

# Breast_Cancer Dataset

In [12]:
cancer = pd.read_csv('D:/Machine_Learning/Cases/Wisconsin/BreastCancer.csv', index_col=0)

le = LabelEncoder()
cancer['Class'] = le.fit_transform(cancer['Class'])

X , y = cancer.drop('Class', axis=1), cancer['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26, stratify= cancer['Class'])

In [13]:
cancer

,Clump,UniCell_Size,Uni_CellShape,MargAdh,SEpith,BareN,BChromatin,NoemN,Mitoses,Class
Code,,,,,,,,,,
61634,5,4,3,1,2,2,2,3,1,0
63375,9,1,2,6,4,10,7,7,2,1
76389,10,4,7,2,2,8,6,1,1,1
95719,6,10,10,10,8,10,7,10,7,1
128059,1,1,1,1,2,5,5,1,1,0
...,...,...,...,...,...,...,...,...,...,...
1369821,10,10,10,10,5,10,10,10,7,1
1371026,5,10,10,10,4,10,5,6,3,1
1371920,5,1,1,1,2,1,3,2,1,0


#### Linear Kernel

In [17]:
Cs=np.linspace(0.001,5,20)
scores=[]
for c in tqdm(Cs):
    svm=SVC(kernel='linear', probability=True, random_state=26, C=c)
    svm.fit(X_train, y_train)
    
    y_pred_prob = svm.predict_proba(X_test)
    
    scores.append([c, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['C', 'log_loss' ])
df_scores.sort_values(['log_loss'], ascending=True).head()

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 24.73it/s]


,C,log_loss
0,0.001000,0.097227
1,0.264105,0.102645
2,0.527211,0.103446
4,1.053421,0.103609
3,0.790316,0.103611


#### Polynomial Kernel

In [20]:
Cs=np.linspace(0.001,5,20)
Ds=[2,3,4]
scores=[]

for c in tqdm(Cs):
    for d in Ds:
        svm=SVC(kernel='poly', probability=True, random_state=26, C=c, degree=d)
        svm.fit(X_train, y_train)
        
        y_pred_prob = svm.predict_proba(X_test)
        
        scores.append([c, d, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['C', 'Degree', 'Log_Loss' ])
df_scores.sort_values(['Log_Loss'], ascending=True).head()

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 13.65it/s]


,C,Degree,Log_Loss
25,2.105842,3,0.104768
28,2.368947,3,0.104804
33,2.895158,2,0.105757
22,1.842737,3,0.105781
31,2.632053,3,0.105833


#### Radial Kernel

In [19]:
Cs=np.linspace(0.001,5,20)
Gs=np.linspace(0.001,5,20)
scores=[]

for c in tqdm(Cs):
    for g in Gs:
        svm=SVC(kernel='rbf', probability=True, random_state=26, C=c, gamma=g)
        svm.fit(X_train, y_train)
        
        y_pred_prob = svm.predict_proba(X_test)
        
        scores.append([c, g, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['C', 'Gamma', 'Log_Loss' ])
df_scores.sort_values(['Log_Loss'], ascending=True).head()

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:23<00:00,  1.16s/it]


,C,Gamma,Log_Loss
260,3.421368,0.001,0.097582
80,1.053421,0.001,0.097644
140,1.842737,0.001,0.098023
40,0.527211,0.001,0.098143
280,3.684474,0.001,0.098186
